In [10]:
!uv sync

Resolved 103 packages in 9ms
Checked 79 packages in 17ms


# 1. Loading and preparation

All experiment code uses reusable modules from `avito_retrieval`. The test set is loaded only for final inference; validation uses `calibration`.

In [11]:
import pandas as pd
from avito_retrieval import load_dataset

In [12]:
DATA_PATH = "candidate_public/candidate_data"

In [13]:
data = load_dataset(DATA_PATH)

/Users/waxyyy99/Code/avito_bootcamp_test/.venv/lib/python3.13/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [14]:
articles = data.articles
calibration = data.calibration
test = data.test

print(f"articles={len(articles)}, calibration={len(calibration)}, test={len(test)}")
articles[["article_id", "clean_title", "document_text"]].head()

articles=793, calibration=500, test=500


,article_id,clean_title,document_text
0,1730,имя или название компании,имя или название компании имя или название ком...
1,1746,"понять, что профиль заблокирован","понять, что профиль заблокирован понять, что п..."
2,1747,не допустить блокировки профиля,не допустить блокировки профиля не допустить б...
3,1774,оставить или удалить профиль,оставить или удалить профиль оставить или удал...
4,1775,удалить профиль,удалить профиль удалить профиль ⚡ удалить проф...


# 2. Lexical baseline

This ranker combines word TF-IDF query-to-query retrieval, character n-gram query-to-query retrieval, and direct word TF-IDF article retrieval. Every fold fits on its own calibration training split.

In [15]:
from avito_retrieval import evaluate_lexical_cv

lexical_result = evaluate_lexical_cv(calibration, articles)
print("fold MAP@10:", [round(score, 4) for score in lexical_result.fold_scores])
print(f"mean MAP@10: {lexical_result.mean_score:.4f}")

fold MAP@10: [0.5416, 0.5611, 0.5575, 0.5863, 0.593]
mean MAP@10: 0.5679


# 3. Local FRIDA embeddings

FRIDA is loaded from `models/frida` with `local_files_only=True`. It uses `paraphrase` for query-to-query retrieval and the asymmetric `search_query` / `search_document` prompts for query-to-article retrieval.

In [16]:
from avito_retrieval import FridaEmbedder

frida = FridaEmbedder(model_path="models/frida", batch_size=8)
sample_vectors = frida.encode(
    ["как вернуть деньги за товар", "когда вернут деньги после возврата"],
    prompt_name="paraphrase",
)
print(f"device={frida.device}, vectors={sample_vectors.shape}")
print(f"cosine similarity={sample_vectors[0] @ sample_vectors[1]:.4f}")

Loading weights: 100%|██████████| 219/219 [00:03<00:00, 63.79it/s] 


device=mps, vectors=(2, 1536)
cosine similarity=0.8408


# 4. Embedding model comparison

All models use the same precomputed calibration queries and the same 5-fold protocol. `max` selects the highest similarity among calibration examples associated with an article. Embeddings are cached locally in `cache/`.

In [17]:
from pathlib import Path
import gc

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from avito_retrieval import evaluate_dense_query_to_query_cv, load_or_create_embeddings

MODEL_CONFIGS = {
    "FRIDA": {"path": "models/frida", "prompt_name": "paraphrase", "cache": "cache/frida_calibration_paraphrase.npz"},
    "Qwen3-Embedding-0.6B": {"path": "models/qwen3-embedding-0.6b", "prompt_name": "query", "cache": "cache/qwen3_0.6b_calibration_query.npz"},
    "BGE-M3": {"path": "models/bge-m3", "prompt_name": None, "cache": "cache/bge_m3_calibration.npz"},
}

comparison = []
for name, config in MODEL_CONFIGS.items():
    model = SentenceTransformer(
        config["path"],
        device="mps" if torch.backends.mps.is_available() else "cpu",
        local_files_only=True,
        model_kwargs={"torch_dtype": torch.float16 if torch.backends.mps.is_available() else torch.float32},
    )
    embeddings = load_or_create_embeddings(
        config["cache"],
        calibration["clean_query"].tolist(),
        lambda texts: model.encode(
            texts,
            prompt_name=config["prompt_name"],
            batch_size=8,
            normalize_embeddings=True,
            convert_to_numpy=True,
            show_progress_bar=False,
        ),
    )
    result = evaluate_dense_query_to_query_cv(embeddings, calibration, aggregation="max")
    comparison.append({"model": name, "MAP@10": result.mean_score, "fold_scores": result.fold_scores})
    del model
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

pd.DataFrame(comparison).sort_values("MAP@10", ascending=False)

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 152.51it/s]


,model,MAP@10,fold_scores
2,BGE-M3,0.598811,"(0.6257460317460317, 0.5931507936507936, 0.595..."
0,FRIDA,0.584629,"(0.5906904761904762, 0.5728373015873016, 0.559..."
1,Qwen3-Embedding-0.6B,0.575373,"(0.5919365079365079, 0.5848055555555556, 0.543..."


# 5. Hybrid fusion

The final ensemble fuses independent rankings with Reciprocal Rank Fusion (RRF): BGE-M3 and FRIDA query-to-query channels, the lexical ranker, and BGE-M3 article chunk retrieval. Query-to-query channels are generated out-of-fold, so validation queries cannot retrieve their own calibration labels. Article chunks are 384 words with a 64-word overlap; title and body chunks are encoded separately and article scores use max pooling.

In [18]:
from sklearn.model_selection import KFold

from avito_retrieval import (
    LexicalRetriever,
    chunk_articles,
    rank_article_scores,
    reciprocal_rank_fusion,
    score_query_to_article,
    score_query_to_query,
)
from avito_retrieval.metrics import mean_average_precision_at_k

article_ids = articles["article_id"].to_numpy()
bge_vectors = np.load("cache/bge_m3_calibration.npz")["embeddings"]
frida_vectors = np.load("cache/frida_calibration_paraphrase.npz")["embeddings"]
chunks = chunk_articles(articles)
chunk_vectors = np.load("cache/bge_m3_article_chunks_384_64.npz")["embeddings"]

article_ranking = rank_article_scores(
    score_query_to_article(bge_vectors, chunk_vectors, chunks.article_ids, article_ids),
    article_ids,
    top_k=100,
)
bge_ranking = np.empty((len(calibration), 79), dtype=np.int64)
frida_ranking = np.empty((len(calibration), 79), dtype=np.int64)
lexical_ranking = np.empty((len(calibration), 100), dtype=np.int64)
folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(calibration))

for train_index, validation_index in folds:
    train = calibration.iloc[train_index]
    validation = calibration.iloc[validation_index]
    bge_ranking[validation_index] = rank_article_scores(
        score_query_to_query(
            bge_vectors[validation_index],
            bge_vectors[train_index],
            train["ground_truth_ids"].tolist(),
            article_ids,
        ),
        article_ids,
        top_k=79,
    )
    frida_ranking[validation_index] = rank_article_scores(
        score_query_to_query(
            frida_vectors[validation_index],
            frida_vectors[train_index],
            train["ground_truth_ids"].tolist(),
            article_ids,
        ),
        article_ids,
        top_k=79,
    )
    lexical = LexicalRetriever().fit(train, articles)
    lexical_ranking[validation_index] = [
        lexical.rank(query, top_k=100) for query in validation["clean_query"]
    ]

fused_ranking = reciprocal_rank_fusion(
    [bge_ranking, frida_ranking, lexical_ranking, article_ranking],
    article_ids,
    weights=(2.0, 1.5, 3.0, 0.5),
    rank_constant=100,
)
fold_scores = [
    mean_average_precision_at_k(
        fused_ranking[validation_index],
        calibration.iloc[validation_index]["ground_truth_ids"],
    )
    for _, validation_index in folds
]
print(f"Hybrid RRF MAP@10: {np.mean(fold_scores):.6f}")
print("Fold scores:", [round(score, 6) for score in fold_scores])

Hybrid RRF MAP@10: 0.648779
Fold scores: [0.644647, 0.65706, 0.63481, 0.65298, 0.654401]
